## Imports

In [1]:
import numpy as np
import scipy
import sklearn
import pandas as pd
import matplotlib as mpl
from matplotlib import pyplot as pyplot
import json
import os
import copy

## Load the data to visualize it

In [2]:
all_data_matrix = pd.read_csv('../Data/raw_data/all_subjects.csv')
subject_1 = all_data_matrix[all_data_matrix['subject']=='A2TQNX64349OZ9:3WSELTNVR4AZUVYAYCSWZIQW0J4ATB']

In [3]:
with open('../Data/raw_data/problems/prb48146_16.json') as prb48146_16:
    problem_1 = json.load(prb48146_16)
    print(problem_1)

{'id': 'prb48146', 'cars': [{'length': 2, 'id': '0', 'orientation': 'vertical', 'position': 16, 'player': False}, {'length': 2, 'id': '2', 'orientation': 'vertical', 'position': 15, 'player': False}, {'length': 2, 'id': '4', 'orientation': 'vertical', 'position': 8, 'player': False}, {'length': 3, 'id': '7', 'orientation': 'vertical', 'position': 17, 'player': False}, {'length': 2, 'id': '6', 'orientation': 'vertical', 'position': 0, 'player': False}, {'length': 3, 'id': '1', 'orientation': 'horizontal', 'position': 3, 'player': False}, {'length': 2, 'id': '3', 'orientation': 'horizontal', 'position': 1, 'player': False}, {'length': 2, 'id': 'r', 'orientation': 'horizontal', 'position': 12, 'player': True}, {'length': 3, 'id': '5', 'orientation': 'horizontal', 'position': 25, 'player': False}]}


## Creating a board/problem object to contain the information in the json file

In [ ]:
# I made a move class just to make my life easier when it comes to playing out a problem
class move:
    def __init__(self,
                 id, # Which car
                 move, # Forward or backward
                 ):
        self.id = id
        self.move = move
    
    def unpack(self):
        return self.id,self.move # take the move and return a list of the items

class problem:
    def __init__(self,json_file):
        with open(json_file) as file:
            problem = json.load(file)
        self.id = problem['id']
        self._cars_tmp = problem['cars']
        self._board = np.zeros(36)
        self._board_2d = self._board.reshape(6,6)

        # Initialize a new cars value which takes the form of dict, so I can access car values easier
        self._cars = {}
        self.history = []
        for i in self._cars_tmp:
            id = i['id']
            if id != 'r':
                id = str(int(id)+1)
            else:
                id = '-1'
            i_orient = i['orientation']
            i_pos = i['position']
            i_len = i['length']
            if i_orient == 'horizontal':
                sq_occ = np.arange(i_pos,i_pos+i_len)
            elif i_orient == 'vertical':
                sq_occ = np.arange(i_pos,i_pos+6*i_len,6)
            self._board[sq_occ] = int(id)

            self._cars[id] = {
                'orientation': i_orient,
                'position': i_pos,
                'length': i_len,
                'sq_occ': sq_occ}
    
    # Print a board (this is just to improve QOL when I'm debugging)
    def display_board_2d(self):
        self._board_2d = self._board.reshape(6,6)
        print('-------------------------')
        for row in self._board_2d:
            print('|',end = '')
            print(*(' ' + str(int(i)) + ' ' if i > -1 else ' r ' for i in row),end = '')
            print('|',end = '\n')
        print('-------------------------')


    # Change the indices of a car based on a move object
    def hyp_move(self,move_obj):
        id,move = move_obj.unpack()
        car = self._cars[id]
        move_dir = car['orientation']
        sq_occ = car['sq_occ']
        sq_change = move*(move_dir=='horizontal') + move*(move_dir=='vertical')*6

        return sq_occ + sq_change
    
    def sqs_passed(self,move_obj):
        id,move = move_obj.unpack()
        car = self._cars[id]
        move_dir = car['orientation']
        sq_occ = car['sq_occ']
        new_occ = self.hyp_move(move_obj)
        temp = np.concatenate((sq_occ,new_occ))
        d1_dir = ((move_dir =='horizontal') + (move_dir == 'vertical')*6)
        sqs_passed = np.arange(np.min(temp),np.max(temp)+d1_dir,d1_dir)
        return sqs_passed
    
    # Check whether a move is legal, return one of three states
    def check_legal(self,move_obj):
        id,move = move_obj.unpack()
        car = self._cars[id]
        move_dir = car['orientation']
        sq_occ = car['sq_occ']
        legal = True
        sqs_passed = self.sqs_passed(move_obj)
        if move_dir != car['orientation']:
            #print('Illegal - car orientation and move direction conflict')
            legal = False
        if move_dir == 'horizontal' and np.any(sqs_passed // 6 != sq_occ[0] // 6):
            #print('Illegal - car went out of bounds')
            legal = False
        elif np.any(sqs_passed < 0) or np.any(sqs_passed > 35):
            #print('Illegal - car went out of bounds')
            legal = False
        elif np.any((self._board[sqs_passed] != 0) == (self._board[sqs_passed] != int(id))):
            legal = 'Blocked'
        return legal
        
    # Check what cars a car is blocked by
    def blocked_by(self,move_obj):
        id,move = move_obj.unpack()
        car = self._cars[id]
        move_dir = car['orientation']
        sq_occ = car['sq_occ']
        blocked = self.check_legal(move_obj)
        if blocked == 'Blocked':
            sqs_passed = self.sqs_passed(move_obj)
            if np.any((self._board[sqs_passed] != 0) == (self._board[sqs_passed] != int(id))):
                blocked_cars_id = np.unique(self._board[sqs_passed][(self._board[sqs_passed] != 0) == (self._board[sqs_passed] != int(id))])
                temp_board = self.get_board().copy()
                temp_board[sqs_passed] = -1
                blocked_spaces_list = []
                for blocked_car in blocked_cars_id:
                    goal_car_mask = (temp_board == -1)
                    blocked_car_mask = (self.get_board() == int(float(blocked_car)))
                    mask = np.logical_and(goal_car_mask,blocked_car_mask)
                    overlapping_sqs = np.argwhere(mask).flatten()
                    blocked_spaces_list.append(overlapping_sqs)

                ids_as_str = [str(i) for i in blocked_cars_id]
                return list(zip(ids_as_str,blocked_spaces_list))
            else:
                return None
    
    # Make a move
    def make_move(self,move_obj):
        id,move = move_obj.unpack()
        car = self._cars[id]
        sq_occ = car['sq_occ']
        legal = self.check_legal(move_obj)
        if legal == True:
            new_occ = self.hyp_move(move_obj)
            car['position'] = new_occ[0]
            car['sq_occ'] = new_occ
            self._board[sq_occ] = 0
            self._board[new_occ] = int(id)
            self.history.append(move_obj)
        else:
            return None
            #print('Illegal move - blocked by: ',*(str(int(float(i))) + ' ' for i in self.blocked_by(move_obj)))
    
    # Undo the last move in the problem history
    def undo_move(self):
        if not self.history:
            print('No moves to undo')
        else:
            last_move = self.history[-1]
            id,move_qt = last_move.unpack()
            self.make_move(move(id,-move_qt))
            self.history.pop()
            self.history.pop()
    
    # Get the dict of cars
    def get_cars(self):
        return self._cars
    
    def all_legal_moves(self):
        legal_moves_list = []
        cars_list = list(self._cars.keys())
        for car in cars_list:
            pos = True
            neg = True
            i = 1
            while pos or neg:
                if pos:
                    potential_move = move(car,i)
                    legal = self.check_legal(potential_move)
                    if legal == True:
                        legal_moves_list.append(potential_move)
                    else:
                        pos = False
                if neg:
                    potential_move = move(car,-i)
                    legal = self.check_legal(potential_move)
                    if legal == True:
                        legal_moves_list.append(potential_move)
                    else:
                        neg = False
                i += 1
        return legal_moves_list
    
    def reset_board(self):
        while self.history:
            self.undo_move()

    def get_board(self):
        return self._board

    """def dist_to_goal(self,dist=0):
        copy = copy.deepcopy(self)
        goal_car = copy._cars['-1']
        goal_car_pos = goal_car['position']
        winning_move = move('-1',4 - (goal_car_pos % 6))
        legal = self.check_legal(winning_move)
        if legal == True:
            dist += 1
            return dist
        else:
            legal_moves = copy.all_legal_moves()
            for i in legal_moves:
                return"""

    def unblocking_moves(self,id,squares):
        pos = True
        neg = True
        i = 1
        car = self._car[id]
        unblocking_moves = [] 
        while pos or neg:
            if pos:
                potential_move = move(car,i)
                sqs_passed = self.sqs_passed(potential_move)
                legal = self.check_legal(potential_move)
                if (legal != False):
                    if np.in1d(sqs_passed,squares):
                        self.unblocking_moves.append(potential_move)
                else:
                    pos = False
            if neg:
                potential_move = move(car,-i)
                sqs_passed = self.sqs_passed(potential_move)
                legal = self.check_legal(potential_move)
                if (legal != False):
                    if np.in1d(sqs_passed,squares):
                        self.unblocking_moves.append(potential_move)
                else:
                    neg = False
            i += 1

        return unblocking_moves


In [32]:
dir_str = '../Data/raw_data/problems'
directory = os.fsencode(dir_str)
problem_list = {}
for file in os.listdir(directory):
    filename = os.fsdecode(file)
    key,extra = filename.split('.')
    unpacked_problem = problem(dir_str + '/' + str(filename))
    problem_list[key] = unpacked_problem

In [33]:
test_board = copy.copy(problem_list['prb46639_16'])

In [34]:
test_board.blocked_by(move('-1',3))

[('5.0', array([16]))]

In [35]:
test_board.make_move(move('-1',1))
legal_moves = test_board.all_legal_moves()
print(*(legal_move.unpack() for legal_move in legal_moves),sep = '\n')
test_board.display_board_2d()

('6', 1)
('6', 2)
('6', 3)
('6', 4)
('5', -1)
('2', -1)
('1', -1)
('1', -2)
('1', -3)
('-1', 1)
('-1', -1)
('7', 1)
('7', 2)
-------------------------
| 6   0   0   0   1   1 |
| 6   7   7   7   0   0 |
| 0   r   r   0   5   0 |
| 0   2   2   2   5   0 |
| 0   0   3   4   5   0 |
| 0   0   3   4   8   8 |
-------------------------


## Making AND/OR Tree

In [ ]:
class OR:
    def __init__(
            self,
            car_id
    ):
        self._car = car_id
        self._prob = None
        self._gamma = None
        self._stop_prob = None
        self._children = []
    
    def set_prob(self,prob):
        self._prob = prob
    
    def set_gamma(self,gamma):
        self._gamma = gamma

    def set_stop_prob(self,stop_prob):
        self._stop_prob = stop_prob
    
    def set_child(self,child):
        self._children.append(child)

    def propagate_prob(self):
        cont = (self._prob - self._stop_prob) / len(self._children)
        for i in self._children:
            i.set_prob(cont)
            i.set_gamma(self._gamma)
            i.set_stop_prob(self._stop_prob)
            i.propagate_prob(cont)

class AND:
    def __init__(
            self,
            move
    ):
        self._move = move
        self._prob = None
        self._gamma = None
        self._stop_prob = None
        self._children = []
    
    def set_prob(self,prob):
        self._prob = prob
    
    def set_child(self,child):
        self._children.append(child)
    
    def set_gamma(self,gamma):
        self._gamma = gamma

    def set_stop_prob(self,stop_prob):
        self._stop_prob = stop_prob
    
    def propagate_prob(self):
        cont = self._prob / len(self._children)
        child_stop_prob = self._gamma*self._stop_prob
        for i in self._children:
            i.set_gamma(self._gamma)
            i.set_stop_prob(child_stop_prob)
            i.propagate_prob(cont)

In [ ]:
class ANDOR:
    def __init__(self,
                 problem):
        self._problem = problem
    
    def maketree(self):
        goal_car = self._problem._cars['-1']
        goal_car_pos = goal_car['position']
        winning_move = move('-1',4 - (goal_car_pos % 6))
        self.tree = OR('-1',None,None,None)
        self.tree.set_child(winning_move)
        blocked_by = self._problem.blocked_by(winning_move)
        sqs_passed = self._problem.sqs_passed(winning_move)
        temp_board = self._problem.get_board().copy()
        temp_board[sqs_passed] = -1
        #print(temp_board.reshape(6,6))
        #self._problem.display_board_2d()
        for blocked_car in blocked_by:
            bl_id, bl_sq = blocked_car
            unblocking_moves = self._problem.unblocking_moves(bl_id,bl_sq)
            OR_Child = OR(bl_id)
            self.tree.set_child(OR_Child)
            for unblocking_move in unblocking_moves:
                if not unblocking_moves:
                    AND_Child = AND(unblocking_move)
                    OR_Child.set_child(AND_Child)
                    

            

Pseudo-code for the tree generation:

def recursive_child_maker(AND_NODE):
    move = AND_NODE.move()
    isblocked = blocked_by(move)
    if isblocked = None:
        return None

    else:
        for block_id, blocked_sq in isblocked:
            AND_NODE.set_child(block_id)
            OR_NODE = AND_NODE.get_child(id)
            unblocking_moves(block_id,blocked_sq)
            for unblocking move in unblocking_moves:
                    recursive_child_maker(unblocking_moves)



In [44]:
test_board_2 = copy.copy(problem_list['prb46639_16'])
test_model = ANDOR(test_board_2)

In [45]:
test_model.maketree()

5.0 [16]
